Warung Suara - Synthetic Voice Dataset Generator (v2 - VITS)
==============================================================
Tahap 1 dari alur dataset (lihat "ALUR DATASET" di bawah). Men-generate
pasangan (audio, label terstruktur) yang dipakai sebagai training data untuk
fine-tuning Whisper pada tahap berikutnya.

KENAPA PINDAH DARI CSM-1B KE VITS (facebook/mms-tts-ind):
  Percobaan awal pakai adapter LoRA `Ellbendls/csm-1b-indonesian-fine-tuned`
  di atas `unsloth/csm-1b` menghasilkan audio yang TIDAK STABIL: sebagian
  0 detik, sebagian cuma 1 suku kata, sebagian isinya silence penuh. Ini
  match dengan bug yang sudah dilaporkan komunitas CSM-1B: model autoregresif
  ini men-generate durasi tetap (~10 detik) terlepas dari panjang teks, dan
  sering "tidak tahu kapan berhenti" - untuk teks pendek, hasilnya audio benar
  diikuti noise/silence. Ditambah adapter Indonesia-nya sendiri cuma dilatih
  100 training steps (sangat minim), jadi kualitasnya memang belum matang.

  `facebook/mms-tts-ind` (arsitektur VITS, dari project Massively Multilingual
  Speech milik Meta AI) dipilih sebagai gantinya karena VITS itu NON-
  AUTOREGRESIF: durasi tiap fonem diprediksi langsung dari struktur teks
  (bukan digenerate token demi token), jadi tidak ada risiko model "kebablasan"
  atau berhenti prematur. Trade-off: cuma 1 suara (bukan 81 seperti CSM), tapi
  untuk kebutuhan sekarang (dataset training Whisper yang butuh STABIL &
  BERSIH) ini prioritas yang lebih tepat dibanding variasi suara.

ALUR DATASET (untuk proposal - bagian Metodologi):
  1. Sumber data: SINTETIK, bukan rekaman asli. Dipilih karena (a) belum ada
     dataset publik voice-note transaksi warung Bahasa Indonesia dengan label
     terstruktur (item/qty/unit/action), dan (b) rulebook AIC mengizinkan
     dataset sintetik selama alur perolehan & preprocessing-nya dijelaskan.
  2. Skema entitas (item, qty, unit, action) didefinisikan lebih dulu, lalu
     kalimat dibuat dari kombinasi template x vocab -> ini yang menjamin
     SETIAP audio yang dihasilkan otomatis punya ground-truth label yang
     akurat (tidak perlu anotasi manual/transkripsi ulang).
  3. Variasi disengaja dimasukkan di beberapa sumbu: gaya kalimat (TEMPLATES,
     sekarang 12 pola), bentuk pengucapan angka (QTY_WORDS: 14 nilai, kata vs
     digit), sinonim aksi (ACTIONS), variasi prosodi (noise_scale VITS), DAN
     augmentasi audio (speed perturbation + noise latar - lihat fungsi
     augment_audio) - supaya model STT tidak overfit ke satu ritme/kondisi
     rekaman yang terlalu bersih dan tidak representatif suasana warung asli.
  4. Tiap kombinasi label (item, qty, action) sekarang menghasilkan BEBERAPA
     kalimat unik berbeda (TEXT_VARIANTS_PER_LABEL_COMBO), bukan cuma 1 -
     jadi variasi tekstual dataset jauh lebih kaya tanpa perlu nambah vocab.
  4. Rendering suara: tiap kalimat teks diucapkan lewat model TTS pretrained
     VITS Indonesian (lihat "PENGGUNAAN MODEL PRETRAINED" di bawah).
  5. Validasi otomatis: tiap audio dicek durasi & energi (RMS) minimum -
     kalau gagal (misal kosongan), otomatis di-retry dengan noise_scale
     berbeda, sampai batas percobaan tertentu, lalu dicatat kalau tetap gagal.
  6. Output: audio .wav + manifest berlabel (JSONL & CSV) -> manifest ini
     yang jadi INPUT LANGSUNG ke script fine-tuning Whisper di tahap
     berikutnya (kolom "text" jadi target transkripsi, kolom lain jadi
     ground-truth untuk evaluasi extractor rule-based/NLP).

PENGGUNAAN MODEL PRETRAINED (untuk kepatuhan rulebook AIC):
  Model `facebook/mms-tts-ind` di sini dipakai APA ADANYA (inference only,
  tanpa fine-tune) HANYA untuk membangkitkan data suara sintetik - bukan
  komponen inti produk Warung Suara. Komponen inti yang wajib di-fine-tune
  sesuai rulebook (poin 10 Ketentuan Khusus: "Model wajib di fine tune
  sesuai dengan inovasi fitur per tim") adalah model Whisper STT yang
  di-fine-tune pakai dataset hasil script ini - dilakukan di script
  terpisah (tahap Minggu 2), bukan di sini.

CATATAN PENTING:
- Script ini butuh akses ke huggingface.co untuk download model. VITS untuk
  1 bahasa jauh lebih kecil (puluhan-ratusan MB) dibanding CSM-1B (~4GB),
  jadi CPU pun sebenarnya cukup layak - tapi GPU tetap lebih cepat.
- Sesuaikan jumlah sampel dan vocab item sesuai kebutuhan.
- Simpan juga versi awal manifest.jsonl/csv di repo (atau ringkasannya) -


In [ ]:
import json
import random
import csv
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
from transformers import VitsModel, AutoTokenizer, set_seed

# 1. KONFIGURASI

In [ ]:
MODEL_ID = "facebook/mms-tts-ind"
OUTPUT_DIR = Path("dataset_warung_suara2")
AUDIO_DIR = OUTPUT_DIR / "audio"
SAMPLES_PER_TEMPLATE_COMBO = 2   # berapa kali tiap kombinasi kalimat diucap ulang (variasi prosodi)
SEED = 42

# Variasi prosodi (noise_scale mengatur variasi ritme/intonasi stochastic
# duration predictor VITS) - dipakai sebagai pengganti variasi speaker_id
# karena model ini cuma 1 suara.
SPEECH_VARIANTS = [
    {"noise_scale": 0.667, "noise_scale_duration": 0.8},   # default
    {"noise_scale": 0.5,   "noise_scale_duration": 0.6},   # lebih datar/cepat
    {"noise_scale": 0.8,   "noise_scale_duration": 1.0},   # lebih ekspresif/lambat
]

# Berapa variasi KALIMAT UNIK (template+qty_word+action_word berbeda) yang
# dibuat untuk tiap kombinasi label (item, qty, action_type). Sebelumnya cuma
# 1 (dipilih acak) - sekarang beberapa, supaya dataset tidak textually sempit.
TEXT_VARIANTS_PER_LABEL_COMBO = 2

MIN_DURATION_SEC = 0.4   # audio di bawah ini dianggap gagal (terlalu pendek/kosong)
MIN_RMS = 0.005          # RMS di bawah ini dianggap silence/gagal
MAX_RETRIES = 3

# Augmentasi audio (diterapkan ke DUPLIKAT tiap sampel yang berhasil, bukan
# menggantikan versi asli) - untuk simulasi kondisi rekaman voice note asli:
# kecepatan bicara sedikit beda & ada noise latar (warung/jalan/HP mic).
ENABLE_AUGMENTATION = True
SPEED_RANGE = (0.92, 1.08)      # variasi tempo bicara +/- 8%
NOISE_SNR_DB_RANGE = (18, 30)   # semakin kecil dB = noise makin kentara

random.seed(SEED)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# 2. VOCAB & SKEMA ENTITAS

In [ ]:
ACTIONS = {
    "keluar": ["laku", "kejual", "terjual", "abis dibeli", "keluar", "diambil pembeli"],
    "masuk":  ["masuk", "stok masuk", "baru dateng", "restock", "nambah stok", "kiriman dateng"],
}

ITEMS = {
    "indomie":           "bungkus",
    "telor":             "kg",
    "gula":              "kg",
    "beras":             "kg",
    "minyak goreng":     "liter",
    "kecap":             "botol",
    "sabun mandi":       "batang",
    "rokok":             "bungkus",
    "aqua gelas":        "dus",
    "gas elpiji":        "tabung",
    "kopi sachet":       "renceng",
    "susu kental manis": "kaleng",
    "mie sedaap":        "bungkus",
    "teh celup":         "kotak",
    "sabun cuci piring": "botol",
    "garam":             "bungkus",
    "tepung terigu":     "kg",
    "saos sambal":       "botol",
    "royco":       "sachet",
    "korek api":         "kotak",
    "tissue":            "pack",
    "air mineral botol": "dus",
    "roti tawar":        "bungkus",
    "margarin":          "bungkus",
    "deterjen":          "bungkus",
}

ITEM_SPOKEN_OVERRIDES = {
    "indomie": "indomi",
}

QTY_WORDS = {
    1:  ["satu", "se", "1"],
    2:  ["dua", "2"],
    3:  ["tiga", "3"],
    4:  ["empat", "4"],
    5:  ["lima", "5"],
    6:  ["enam", "6"],
    7:  ["tujuh", "7"],
    8:  ["delapan", "8"],
    10: ["sepuluh", "10"],
    12: ["dua belas", "12"],
    15: ["lima belas", "15"],
    20: ["dua puluh", "20"],
    25: ["dua puluh lima", "25"],
    50: ["lima puluh", "50"],
}

# 3. TEMPLATE KALIMAT (variasi gaya ngomong pedagang)

In [ ]:
TEMPLATES = [
    "{action} {item} {qty} {unit}",
    "{item} {action} {qty} {unit}",
    "tadi {action} {item} {qty} {unit}",
    "{action} {item} {qty} {unit} ya",
    "eh {item} {action} {qty} {unit}",
    "barusan {action} {item} {qty} {unit}",
    "{qty} {unit} {item} {action}",
    "catet ya {item} {action} {qty} {unit}",
    "{action} tuh {item} {qty} {unit}",
    "iya {item} {qty} {unit} {action}",
    "oh iya {action} {item} {qty} {unit} tadi",
    "{item} nya {action} {qty} {unit}",
]


def build_dataset_entries():
    """
    Preprocessing tahap 1: kombinasi (item x qty x action) -> teks + label.

    Untuk tiap kombinasi label (item, qty, action_type), dibuat BEBERAPA
    variasi kalimat unik (TEXT_VARIANTS_PER_LABEL_COMBO) dengan template,
    kata qty, dan sinonim aksi yang berbeda-beda - bukan cuma 1 kalimat
    acak seperti versi awal. Ini memperkaya variasi TEKSTUAL dataset tanpa
    perlu menambah vocab item/qty itu sendiri.

    Tiap entry punya "text" (kalimat yang akan diucapkan TTS) dan "label"
    (ground-truth terstruktur: item, qty, unit, action). Karena label
    dibuat BERSAMAAN dengan teks (bukan diekstrak belakangan), tidak ada
    risiko label salah/hasil anotasi manual yang bias.
    """
    entries = []
    for item, unit in ITEMS.items():
        for qty, qty_variants in QTY_WORDS.items():
            for action_type, action_variants in ACTIONS.items():
                seen_texts = set()
                attempts = 0
                # generate beberapa variasi kalimat unik untuk kombinasi label ini
                while len(seen_texts) < TEXT_VARIANTS_PER_LABEL_COMBO and attempts < 10:
                    attempts += 1
                    template = random.choice(TEMPLATES)
                    qty_word = random.choice(qty_variants)
                    action_word = random.choice(action_variants)

                    unit_text = unit
                    if qty_word == "se":
                        unit_text = "kilo" if unit == "kg" else unit

                    text = template.format(
                        action=action_word, item=item, qty=qty_word, unit=unit_text
                    ).strip()

                    if text in seen_texts:
                        continue
                    seen_texts.add(text)

                    entries.append({
                        "text": text,
                        "label": {
                            "item": item,
                            "qty": qty,
                            "unit": unit,
                            "action": action_type,
                        },
                    })
    return entries

# 4. LOAD MODEL TTS

In [ ]:
def load_tts():
    """
    Load model VITS pretrained (inference only, tidak di-fine-tune).
    Dipakai murni sebagai "mesin perekam suara sintetik" untuk membangkitkan
    dataset - bukan bagian dari arsitektur inti produk Warung Suara.
    """
    print(f"Loading model {MODEL_ID} ...")
    model = VitsModel.from_pretrained(MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Model loaded on {device}, sample rate: {model.config.sampling_rate}")
    return model, tokenizer, device


def synthesize(model, tokenizer, device, text: str, variant: dict, seed: int) -> np.ndarray:
    """
    Render 1 kalimat teks -> audio. `variant` mengatur noise_scale/
    noise_scale_duration (variasi prosodi), `seed` memastikan tiap sampel
    tetap reproducible meski ada elemen stokastik di duration predictor.
    """
    set_seed(seed)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(
            **inputs,
            noise_scale=variant["noise_scale"],
            noise_scale_duration=variant["noise_scale_duration"],
        ).waveform
    return output[0].float().cpu().numpy()


def is_valid_audio(audio: np.ndarray, sample_rate: int) -> bool:
    """Validasi otomatis: tolak audio yang terlalu pendek atau nyaris silent."""
    duration = len(audio) / sample_rate
    rms = float(np.sqrt(np.mean(audio ** 2))) if len(audio) > 0 else 0.0
    return duration >= MIN_DURATION_SEC and rms >= MIN_RMS


def augment_audio(audio: np.ndarray, sample_rate: int, seed: int) -> np.ndarray:
    """
    Augmentasi audio ringan untuk mensimulasikan kondisi voice note asli:
    - speed perturbation: tempo bicara pedagang bervariasi orang ke orang
    - noise latar (Gaussian, disamakan skalanya lewat target SNR): simulasi
      suara warung/jalan/kualitas mic HP yang tidak pernah benar-benar bersih

    Ini membuat model STT nanti lebih robust ke kondisi dunia nyata,
    dibanding kalau seluruh dataset training-nya "bersih sempurna" dari TTS.
    """
    rng = np.random.RandomState(seed)

    # 1. Speed perturbation via resampling sederhana
    speed = rng.uniform(*SPEED_RANGE)
    n_new = int(len(audio) / speed)
    x_old = np.linspace(0, 1, len(audio))
    x_new = np.linspace(0, 1, n_new)
    audio_speed = np.interp(x_new, x_old, audio).astype(np.float32)

    # 2. Tambah noise latar sesuai target SNR (dB)
    snr_db = rng.uniform(*NOISE_SNR_DB_RANGE)
    signal_power = np.mean(audio_speed ** 2) + 1e-10
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), size=audio_speed.shape).astype(np.float32)
    audio_aug = audio_speed + noise

    # clip supaya tidak melebihi rentang valid audio
    audio_aug = np.clip(audio_aug, -1.0, 1.0)
    return audio_aug

# 5. MAIN GENERATION LOOP

In [ ]:
def main():
    entries = build_dataset_entries()
    n_base = len(entries)
    n_total_planned = n_base * SAMPLES_PER_TEMPLATE_COMBO
    if ENABLE_AUGMENTATION:
        n_total_planned *= 2
    print(f"Total kombinasi kalimat unik: {n_base}")
    print(f"Perkiraan total sampel (termasuk augmentasi): {n_total_planned}")

    model, tokenizer, device = load_tts()
    sample_rate = model.config.sampling_rate

    manifest = []
    failed = []
    idx = 0
    for entry in entries:
        for rep in range(SAMPLES_PER_TEMPLATE_COMBO):
            filename = f"sample_{idx:05d}.wav"
            filepath = AUDIO_DIR / filename

            success = False
            base_audio = None
            for attempt in range(MAX_RETRIES):
                variant = SPEECH_VARIANTS[(rep + attempt) % len(SPEECH_VARIANTS)]
                seed = SEED + idx * 10 + attempt
                try:
                    audio = synthesize(model, tokenizer, device, entry["text"], variant, seed)
                except Exception as e:
                    print(f"  [error] '{entry['text']}' attempt {attempt}: {e}")
                    continue

                if is_valid_audio(audio, sample_rate):
                    sf.write(str(filepath), audio, sample_rate)
                    manifest.append({
                        "audio_path": str(filepath.as_posix()),
                        "text": entry["text"],
                        "noise_scale": variant["noise_scale"],
                        "augmented": False,
                        **entry["label"],
                    })
                    base_audio = audio
                    success = True
                    break
                else:
                    print(f"  [retry] '{entry['text']}' attempt {attempt} gagal validasi "
                          f"(durasi/energi terlalu rendah)")

            if not success:
                failed.append(entry["text"])
                print(f"  [skip] gagal setelah {MAX_RETRIES}x percobaan: '{entry['text']}'")
                idx += 1
                continue

            idx += 1

            # duplikat versi augmentasi (speed + noise) dari sampel yang berhasil
            if ENABLE_AUGMENTATION:
                aug_filename = f"sample_{idx:05d}.wav"
                aug_filepath = AUDIO_DIR / aug_filename
                audio_aug = augment_audio(base_audio, sample_rate, seed=SEED + idx * 10)
                sf.write(str(aug_filepath), audio_aug, sample_rate)
                manifest.append({
                    "audio_path": str(aug_filepath.as_posix()),
                    "text": entry["text"],
                    "noise_scale": variant["noise_scale"],
                    "augmented": True,
                    **entry["label"],
                })
                idx += 1

            if idx % 50 == 0:
                print(f"  {idx} sampel diproses... ({len(manifest)} berhasil, {len(failed)} gagal)")

    jsonl_path = OUTPUT_DIR / "manifest.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in manifest:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    csv_path = OUTPUT_DIR / "manifest.csv"
    if manifest:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
            writer.writeheader()
            writer.writerows(manifest)

    print(f"\nSelesai. {len(manifest)} sampel audio+label tersimpan di '{OUTPUT_DIR}/'")
    print(f"  - Audio  : {AUDIO_DIR}/")
    print(f"  - Label  : {jsonl_path} & {csv_path}")
    if failed:
        print(f"  - Gagal total: {len(failed)} kalimat (lihat log [skip] di atas)")


if __name__ == "__main__":
    main()

Total kombinasi kalimat unik: 1400
Perkiraan total sampel (termasuk augmentasi): 5600
Loading model facebook/mms-tts-ind ...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Model loaded on cuda, sample rate: 16000
  50 sampel diproses... (50 berhasil, 0 gagal)
  100 sampel diproses... (100 berhasil, 0 gagal)
  150 sampel diproses... (150 berhasil, 0 gagal)
  200 sampel diproses... (200 berhasil, 0 gagal)
  250 sampel diproses... (250 berhasil, 0 gagal)
  300 sampel diproses... (300 berhasil, 0 gagal)
  350 sampel diproses... (350 berhasil, 0 gagal)
  400 sampel diproses... (400 berhasil, 0 gagal)
  450 sampel diproses... (450 berhasil, 0 gagal)
  500 sampel diproses... (500 berhasil, 0 gagal)
  550 sampel diproses... (550 berhasil, 0 gagal)
  600 sampel diproses... (600 berhasil, 0 gagal)
  650 sampel diproses... (650 berhasil, 0 gagal)
  700 sampel diproses... (700 berhasil, 0 gagal)
  750 sampel diproses... (750 berhasil, 0 gagal)
  800 sampel diproses... (800 berhasil, 0 gagal)
  850 sampel diproses... (850 berhasil, 0 gagal)
  900 sampel diproses... (900 berhasil, 0 gagal)
  950 sampel diproses... (950 berhasil, 0 gagal)
  1000 sampel diproses... (100

In [ ]:
!zip -r dataset.zip /content/dataset_warung_suara2/

Output streaming akan dipotong hingga 5000 baris terakhir.
  adding: content/dataset_warung_suara2/audio/sample_02955.wav (deflated 13%)
  adding: content/dataset_warung_suara2/audio/sample_00094.wav (deflated 25%)
  adding: content/dataset_warung_suara2/audio/sample_00180.wav (deflated 16%)
  adding: content/dataset_warung_suara2/audio/sample_05584.wav (deflated 24%)
  adding: content/dataset_warung_suara2/audio/sample_02278.wav (deflated 20%)
  adding: content/dataset_warung_suara2/audio/sample_04899.wav (deflated 9%)
  adding: content/dataset_warung_suara2/audio/sample_04368.wav (deflated 22%)
  adding: content/dataset_warung_suara2/audio/sample_02771.wav (deflated 10%)
  adding: content/dataset_warung_suara2/audio/sample_02854.wav (deflated 24%)
  adding: content/dataset_warung_suara2/audio/sample_02752.wav (deflated 21%)
  adding: content/dataset_warung_suara2/audio/sample_04859.wav (deflated 10%)
  adding: content/dataset_warung_suara2/audio/sample_03561.wav (deflated 9%)
  addin